<a href="https://colab.research.google.com/github/OdysseusPolymetis/enexdi_prep_2026/blob/main/4_transformers_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **A Simple Introduction to Transformers for the Humanities and Social Sciences**
---
This notebook offers a very simple introduction to Transformer-based models using Hugging Face’s `transformers` library.

The goal is not to train a model, but to see what can be done with already trained models.

We mainly use default implementations with `pipeline()`.


## 1. Installation

In [ ]:
!pip -q install transformers sentencepiece accelerate pandas scikit-learn matplotlib

## 2. Imports and configuration

We simply tell `transformers` to use the GPU if one is available.

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForMaskedLM

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

device = 0 if torch.cuda.is_available() else -1


## 3. Loading a multilingual model: XLM-RoBERTa

We start by loading **XLM-RoBERTa**, a multilingual model.

It can be used directly for a **fill-mask** task: we give the model a sentence with a masked word, and it suggests likely completions.

The mask used by XLM-RoBERTa is:

```text
<mask>
```

In [ ]:
modele_xlmr = "FacebookAI/xlm-roberta-base"

tokenizer_xlmr = AutoTokenizer.from_pretrained(modele_xlmr)
model_xlmr = AutoModelForMaskedLM.from_pretrained(modele_xlmr)

fill_mask = pipeline(
    "fill-mask",
    model=model_xlmr,
    tokenizer=tokenizer_xlmr,
    device=device
)

print("Modèle chargé :", modele_xlmr)

## 4. Task 1 — Fill-mask

The model must complete a sentence in which one word has been replaced by `<mask>`.

Try modifying the examples.

In [ ]:
phrase = "Paris est la capitale de la <mask>."

fill_mask(phrase, top_k=10)

## 5. Exercise — Comparing several masked sentences

**Instructions:** modify the sentences below, then observe the answers.

Possible questions:

- are the answers relevant?
- do the results change depending on the wording?
- do the results change depending on the language?
- what assumptions does the model seem to have learned?

In [ ]:
phrases = [
    "La Révolution française commence en <mask>.",
    "Le personnage principal du roman est un <mask>.",
    "The capital of France is <mask>.",
    "La ville de Lyon se trouve en <mask>."
]

for phrase in phrases:
    print("\nPhrase :", phrase)
    resultats = fill_mask(phrase, top_k=5)
    for r in resultats:
        print(r["token_str"], "→", round(r["score"], 4))

## 6. Task 2 — Zero-shot classification

*Zero-shot* classification makes it possible to classify a text into categories that we define ourselves, without training the model on our own data.

Note: this task requires a model specialized in textual inference (*NLI*).  
We therefore load a second multilingual model adapted to this task.

In [ ]:
modele_zero_shot = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

zero_shot = pipeline(
    "zero-shot-classification",
    model=modele_zero_shot,
    device=device
)

print("Modèle chargé :", modele_zero_shot)

## 7. Classifying a text with chosen categories

We provide:

- a text;
- a list of possible labels.

The model returns a score for each label.

In [ ]:
texte = """
Le gouvernement annonce une nouvelle réforme visant à modifier l'organisation des universités
et le financement de la recherche publique.
"""

etiquettes = [
    "politique",
    "économie",
    "religion",
    "littérature",
    "science",
    "éducation"
]

zero_shot(texte, etiquettes)

## 8. Classifying a text with several possible labels

With `multi_label=True`, several categories can be considered relevant at the same time.

In [ ]:
zero_shot(
    texte,
    etiquettes,
    multi_label=True
)

## 9. Exercise — Classifying several short texts

Here is a small fictional corpus.

**Instructions:** modify the texts or the labels to adapt them to your own field: history, literature, sociology, the press, archives, etc.

In [ ]:
corpus = [
    "Le roi reçoit les ambassadeurs dans la grande salle du palais.",
    "Le narrateur décrit longuement les sentiments amoureux du personnage.",
    "Les ouvriers se mettent en grève pour demander une augmentation des salaires.",
    "Le savant observe les astres et rédige un traité sur le mouvement des planètes.",
    "Le sermon insiste sur la faute, le pardon et le salut des âmes."
]

etiquettes = [
    "politique",
    "amour",
    "travail",
    "science",
    "religion"
]

resultats = []

for texte in corpus:
    prediction = zero_shot(texte, etiquettes)

    resultats.append({
        "texte": texte,
        "meilleure_etiquette": prediction["labels"][0],
        "score": prediction["scores"][0]
    })

df = pd.DataFrame(resultats)
df

## 10. Exercise — Changing the labels

Classification strongly depends on the proposed labels.

Try replacing the labels below with more fine-grained categories.

In [ ]:
nouvelles_etiquettes = [
    "pouvoir royal",
    "sentiment amoureux",
    "conflit social",
    "savoir scientifique",
    "discours religieux"
]

resultats = []

for texte in corpus:
    prediction = zero_shot(texte, nouvelles_etiquettes)

    resultats.append({
        "texte": texte,
        "meilleure_etiquette": prediction["labels"][0],
        "score": prediction["scores"][0]
    })

pd.DataFrame(resultats)

## 11. Task 3 — Representing sentences as vectors

A Transformer model can also produce vectors.

Here, we use the same XLM-RoBERTa model loaded above.

The idea is simple:

1. we transform a sentence into tokens;
2. the model produces a vector for each token;
3. we compute an average to obtain a sentence vector.

This is not the best possible method for every use case, but it is enough to understand the principle.

In [ ]:
model_xlmr.to("cuda" if torch.cuda.is_available() else "cpu")

def vectoriser_phrase(phrase):
    inputs = tokenizer_xlmr(
        phrase,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_xlmr.roberta(**inputs)

    embeddings_tokens = outputs.last_hidden_state[0]
    attention_mask = inputs["attention_mask"][0]

    embeddings_tokens = embeddings_tokens[attention_mask == 1]

    vecteur_phrase = embeddings_tokens.mean(dim=0)

    return vecteur_phrase.cpu().numpy()

## 12. Comparing two sentences

The closer the score is to 1, the closer the two sentences are in the model’s vector space.


In [ ]:
phrase_1 = "Le roi gouverne le royaume."
phrase_2 = "Le souverain dirige le pays."

v1 = vectoriser_phrase(phrase_1)
v2 = vectoriser_phrase(phrase_2)

score = cosine_similarity([v1], [v2])[0][0]

print("Similarité :", score)

## 13. Exercise — Comparing several sentences

Modify the sentences and observe the scores.


In [ ]:
phrases_a_comparer = [
    "Le roi gouverne le royaume.",
    "Le souverain dirige le pays.",
    "La jeune femme écrit une lettre d'amour.",
    "Les ouvriers réclament de meilleurs salaires.",
    "Le chercheur analyse des données linguistiques."
]

vecteurs = [vectoriser_phrase(p) for p in phrases_a_comparer]

matrice_similarite = cosine_similarity(vecteurs)

df_sim = pd.DataFrame(
    matrice_similarite,
    index=phrases_a_comparer,
    columns=phrases_a_comparer
)

df_sim

## 14. Visualizing sentences in two dimensions

We reduce the vectors to two dimensions using PCA in order to produce a simple visualization.


In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(vecteurs)

plt.figure(figsize=(8, 6))
plt.scatter(coords[:, 0], coords[:, 1])

for phrase, x, y in zip(phrases_a_comparer, coords[:, 0], coords[:, 1]):
    plt.text(x, y, phrase[:35] + "...", fontsize=9)

plt.title("Projection simple de phrases")
plt.xlabel("Axe 1")
plt.ylabel("Axe 2")
plt.show()

## 15. Mini-activity for the Humanities and Social Sciences — Observing the effects of wording

Models do not read texts like humans.  
They respond on the basis of regularities learned from very large corpora.

**Instructions:** test several similar formulations.


In [ ]:
formulations = [
    "Ce texte parle de la Révolution française.",
    "Ce document évoque des événements politiques en France.",
    "Cette source historique décrit une période de crise sociale.",
    "Le passage concerne les sentiments d'un personnage romanesque."
]

etiquettes = [
    "histoire politique",
    "histoire sociale",
    "analyse littéraire",
    "histoire religieuse"
]

for texte in formulations:
    prediction = zero_shot(texte, etiquettes)
    print("\nTexte :", texte)
    print("Classe proposée :", prediction["labels"][0])
    print("Score :", round(prediction["scores"][0], 4))

## 16. Token classification


In [ ]:
ner = pipeline(
    "token-classification",
    model="Davlan/xlm-roberta-base-ner-hrl",
    aggregation_strategy="simple",
    device=device
)

texte = """
Victor Hugo naît à Besançon en 1802. Il séjourne à Paris et publie Notre-Dame de Paris en 1831.
"""

ner(texte)

## 17. Sentiment analysis

In [ ]:
sentiment = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=device
)

phrases = [
    "Ce roman est magnifique.",
    "Cette situation est catastrophique.",
    "Voilà une décision admirablement absurde."
]

sentiment(phrases)